# Credit Risk Detection — Full MLOps Pipeline

**University MLOps Group 10**

This notebook runs the complete credit risk detection pipeline end-to-end


> **Note**: The data pipeline (PDF extraction, label building, embedding generation) has already been run and the outputs are committed to the repo under `data/processed/`, `data/embeddings/`, `data/labels/`, and `data/finetune/`. This notebook skips those steps and goes straight to inference.

---
## Section 1: Setup

Check the GPU, clone the repository, install all Python dependencies, and configure environment variables.


In [1]:
# Verify GPU availability — you should see a T4 (or better)
!nvidia-smi

Sun Apr 26 14:46:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import os


GITHUB_REPO_URL = "https://github.com/pankti0/MLOps-group-10"
USE_MANUAL_UPLOAD = False  # Set True to skip cloning and upload manually

REPO_DIR = "/content/MLOps-group-10"

if USE_MANUAL_UPLOAD:
    print("Manual upload mode selected.")
    print("Please upload the repo zip via Files panel and unzip to /content/MLOps-group-10")
    print("  !unzip /content/MLOps-group-10.zip -d /content/")
elif not os.path.isdir(REPO_DIR):
    print(f"Cloning {GITHUB_REPO_URL} ...")
    !git clone {GITHUB_REPO_URL} {REPO_DIR}
    print("Clone complete.")
else:
    print(f"Repo already exists at {REPO_DIR}, pulling latest...")
    !git -C {REPO_DIR} pull

Cloning https://github.com/pankti0/MLOps-group-10 ...
Cloning into '/content/MLOps-group-10'...
remote: Enumerating objects: 8422, done.
remote: Counting objects: 100% (316/316), done.
remote: Compressing objects: 100% (183/183), done.
remote: Total 8422 (delta 165), reused 258 (delta 118), pack-reused 8106 (from 2)
Receiving objects: 100% (8422/8422), 151.79 MiB | 13.43 MiB/s, done.
Resolving deltas: 100% (1350/1350), done.
Updating files: 100% (7769/7769), done.
Clone complete.


In [3]:
# Install all project dependencies.
print("Installing dependencies from requirements.txt ...")
!pip install -q -r /content/MLOps-group-10/requirements.txt

# Ensure bitsandbytes is the CUDA-enabled version
!pip install -q bitsandbytes>=0.41.0

print("All dependencies installed.")

Installing dependencies from requirements.txt ...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 6.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 61.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 131.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 87.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 55.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 116.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 93.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 112.9 MB/s eta 0:00:00
 

In [4]:

# Configure environment variables.
#


try:
    from google.colab import userdata
    # Try loading from Colab Secrets
    HF_TOKEN = userdata.get('HF_TOKEN')
    WANDB_API_KEY = userdata.get('WANDB_API_KEY')
    print("Loaded secrets from Colab Secrets vault.")
except Exception:
    # Fallback: assign directly
    HF_TOKEN = "hf_YOUR_TOKEN_HERE"
    WANDB_API_KEY = "YOUR_WANDB_API_KEY_HERE"
    print("Using directly-assigned API keys (Colab Secrets not available).")

os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
os.environ["WANDB_API_KEY"] = WANDB_API_KEY


env_content = f"""HF_TOKEN={HF_TOKEN}
HUGGING_FACE_HUB_TOKEN={HF_TOKEN}
WANDB_API_KEY={WANDB_API_KEY}
"""
with open("/content/MLOps-group-10/.env", "w") as f:
    f.write(env_content)

print("Environment variables configured.")

Loaded secrets from Colab Secrets vault.
Environment variables configured.


In [5]:
%cd MLOps-group-10

/content/MLOps-group-10


---
## Section 2: Weights & Biases Login

All training runs, inference runs, and evaluation metrics are logged to W&B for experiment tracking.
If you haven't already, create a free account at [wandb.ai](https://wandb.ai).

In [6]:
import wandb

# Log in using the API key set in Section 1.
# If WANDB_API_KEY env var is set, this proceeds non-interactively.
wandb.login()
print(f"W&B version: {wandb.__version__}")
print("Logged in to Weights & Biases successfully.")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: pankti0 (pankti0-singapore-university-of-technology-and-design) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B version: 0.25.1
Logged in to Weights & Biases successfully.


---
## Section 3: Verify Pre-Processed Data

The following data artefacts were generated by the data pipeline and are committed to the repo.
This cell verifies they all exist before running inference:

- `data/processed/` — per-company JSON files with extracted 10-K sections
- `data/labels/company_labels.csv` — ground-truth credit risk labels
- `data/embeddings/faiss.index` + `chunk_metadata.json` — vector store for RAG
- `data/finetune/train.jsonl`, `val.jsonl`, `test.jsonl` — fine-tuning data

In [7]:
import os
import pandas as pd

REPO_ROOT = "."  # or the absolute path to your repo

required_paths = {
    "Labels CSV"     : f"{REPO_ROOT}/data/labels/company_labels.csv",
    "FAISS index"    : f"{REPO_ROOT}/data/embeddings/faiss.index",
    "Chunk metadata" : f"{REPO_ROOT}/data/embeddings/chunk_metadata.json",
}

# Check cross-validation folds
folds = [1, 2, 3, 4, 5]
for fold in folds:
    for split in ["train", "val", "test"]:
        path = f"{REPO_ROOT}/data/finetune/fold_{fold}/{split}.jsonl"
        required_paths[f"Fold {fold} {split.capitalize()} JSONL"] = path

all_ok = True
for name, path in required_paths.items():
    exists = os.path.isfile(path)
    status = "OK" if exists else "MISSING"
    print(f"  [{status}] {name}: {path}")
    if not exists:
        all_ok = False

processed_dir = f"{REPO_ROOT}/data/processed/"
processed_files = [f for f in os.listdir(processed_dir) if f.endswith(".json") and "sections" in f]
print(f"\n  [OK] Processed sections: {len(processed_files)} companies")
for f in sorted(processed_files):
    print(f"       {f}")

if all_ok:
    print("\nAll required data files present — ready to run inference.")
else:
    print("\nWARNING: Some files are missing. Re-run the data pipeline scripts if needed.")

  [OK] Labels CSV: ./data/labels/company_labels.csv
  [OK] FAISS index: ./data/embeddings/faiss.index
  [OK] Chunk metadata: ./data/embeddings/chunk_metadata.json
  [OK] Fold 1 Train JSONL: ./data/finetune/fold_1/train.jsonl
  [OK] Fold 1 Val JSONL: ./data/finetune/fold_1/val.jsonl
  [OK] Fold 1 Test JSONL: ./data/finetune/fold_1/test.jsonl
  [OK] Fold 2 Train JSONL: ./data/finetune/fold_2/train.jsonl
  [OK] Fold 2 Val JSONL: ./data/finetune/fold_2/val.jsonl
  [OK] Fold 2 Test JSONL: ./data/finetune/fold_2/test.jsonl
  [OK] Fold 3 Train JSONL: ./data/finetune/fold_3/train.jsonl
  [OK] Fold 3 Val JSONL: ./data/finetune/fold_3/val.jsonl
  [OK] Fold 3 Test JSONL: ./data/finetune/fold_3/test.jsonl
  [OK] Fold 4 Train JSONL: ./data/finetune/fold_4/train.jsonl
  [OK] Fold 4 Val JSONL: ./data/finetune/fold_4/val.jsonl
  [OK] Fold 4 Test JSONL: ./data/finetune/fold_4/test.jsonl
  [OK] Fold 5 Train JSONL: ./data/finetune/fold_5/train.jsonl
  [OK] Fold 5 Val JSONL: ./data/finetune/fold_5/val.jso

In [7]:
# Display company labels — these are the ground-truth credit risk labels
import pandas as pd
from IPython.display import display

labels_df = pd.read_csv("/content/MLOps-group-10/data/labels/company_labels.csv")

def style_risk(val):
    colors = {"low": "#d4edda", "medium": "#fff3cd", "high": "#f8d7da"}
    return f"background-color: {colors.get(str(val).lower(), '')}"

print(f"Dataset: {len(labels_df)} companies")
display(
    labels_df.style
    .applymap(style_risk, subset=["risk_category"])
    .format({"risk_score": "{:.0f}"})
    .set_caption("Company Credit Risk Labels (Ground Truth)")
)

Dataset: 32 companies


/tmp/ipykernel_640/669797996.py:14: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  .applymap(style_risk, subset=["risk_category"])


,ticker,company_name,pdf_path,label,risk_category,risk_score
0,AAPL,Apple,10-k forms/Apple_2025_annual_report.pdf,0,low,15
1,JNJ,Johnson & Johnson,10-k forms/Johnson & Johnson_2024_annual_report.pdf,0,low,20
2,DAL,Delta Air Lines,10-k forms/Delta Air Lines_2024_annual_report.pdf,0,medium,45
3,T,AT&T,10-k forms/AT&T_2024_annual_report.pdf,0,medium,55
4,VZ,Verizon,10-k forms/Verizon Communication_2025_annual_report.pdf,0,medium,50
5,F,Ford Motor Company,10-k forms/Ford Motor Company_2024_annual_report.pdf,0,medium,60
6,AAL,American Airlines,10-k forms/American Airlines_2024_annual_report.pdf,1,high,80
7,GME,GameStop,10-k forms/GameStop_2024_annual_report.pdf,1,high,75
8,AMC,AMC Entertainment,10-k forms/AMC Entertainment_2024_annual_report.pdf,1,high,90
9,GOOGL,Alphabet,10-k forms/Alphabet_2025_annual_report.pdf,0,low,10


In [10]:
# Show FAISS index statistics
import json
import faiss

index = faiss.read_index("/content/MLOps-group-10/data/embeddings/faiss.index")
with open("/content/MLOps-group-10/data/embeddings/chunk_metadata.json") as f:
    metadata = json.load(f)

print("FAISS Index Statistics")
print(f"  Total vectors : {index.ntotal}")
print(f"  Dimension     : {index.d}")
print(f"  Metadata chunks: {len(metadata)}")

# Show a sample chunk
if metadata:
    sample = metadata[0]
    print(f"\nSample chunk keys: {list(sample.keys())}")
    print(f"  ticker : {sample.get('ticker', 'N/A')}")
    print(f"  section: {sample.get('section', 'N/A')}")
    print(f"  text   : {str(sample.get('text', ''))[:120]}...")

FAISS Index Statistics
  Total vectors : 1272
  Dimension     : 768
  Metadata chunks: 1272

Sample chunk keys: ['text', 'chunk_id', 'metadata', 'char_start', 'char_end']
  ticker : N/A
  section: N/A
  text   : 

Item 1A. 
Risk Factors
The following summarizes factors that could have a material adverse effect on the Company’s bus...


---
## Section 4: Baseline Inference

Run credit risk inference using **direct prompting** (no retrieval, no fine-tuning).

- Model: `mistralai/Mistral-7B-Instruct-v0.2`
- Quantization: 4-bit NF4 via `bitsandbytes`
- The model reads raw 10-K section text and outputs a risk score + label



## ⚡️ Standardized Stratified K-Fold Cross-Validation (NEW)

All experiments now use **stratified k-fold cross-validation** for robust, fair, and reproducible evaluation.

- Use the `--fold` and `--split` arguments in all inference commands.

- Results are saved as `results/{approach}_fold{N}_{split}.csv`.

- Update all result display code to use the correct fold/split files.

You can loop over all folds for full evaluation.

In [12]:
!git pull

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 761 bytes | 761.00 KiB/s, done.
From https://github.com/pankti0/MLOps-group-10
   83229c0..8132893  main       -> origin/main
Updating 83229c0..8132893
Fast-forward
 scripts/train_lora.py | 32 ++++++++++++++++++++++++++------
 1 file changed, 26 insertions(+), 6 deletions(-)


In [11]:
# Run baseline inference for all folds (1-5) on the test split
for fold in range(1, 6):
    print(f"Running baseline inference for fold {fold} (test split)...")
    !python3 scripts/run_inference_all.py --approach baseline --fold {fold} --split test

Running baseline inference for fold 1 (test split)...
03:07:32 [INFO] run_inference_all: Starting inference: approach=baseline, mock=False.
03:07:32 [INFO] numexpr.utils: NumExpr defaulting to 2 threads.
03:07:33 [INFO] run_inference_all: Loaded 7 companies for fold 1 split 'test'.
03:07:35 [INFO] run_inference_all: Loading model 'mistralai/Mistral-7B-Instruct-v0.2' (quantize=True).
03:07:41 [INFO] src.models.base_loader: HF_TOKEN found — using authenticated HuggingFace download.
03:07:41 [INFO] src.models.base_loader: Loading tokenizer for 'mistralai/Mistral-7B-Instruct-v0.2'.
03:07:41 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
03:07:41 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/mistralai/Mistral-7B-Instruct-v0.2/63a8b081895390a26e140280378bc85ec8bce07a/config.json "HTTP/1.1 200 OK"
03:07:41 [INFO] httpx: HTTP Request: HEAD https://huggingfac

In [8]:
import pandas as pd
import glob

files = glob.glob("/content/MLOps-group-10/data/results/baseline_fold*_test.csv")
df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

print("Rows:", len(df))
print("\nPredicted label counts:")
print(df["predicted_label"].value_counts(dropna=False))

print("\nMax score:", df["predicted_score"].max())

print("\nAny positives?")
print(df[df["predicted_label"] == 1][
    ["ticker","company_name","predicted_score","risk_level"]
])

Rows: 32

Predicted label counts:
predicted_label
0    32
Name: count, dtype: int64

Max score: 50.0

Any positives?
Empty DataFrame
Columns: [ticker, company_name, predicted_score, risk_level]
Index: []


In [12]:
import os
import glob
import pandas as pd
from IPython.display import display

def style_risk_level(val):
    """Color cells based on risk level."""
    color = {
        "low": "background-color: #c6efce; color: #006100",      # green
        "medium": "background-color: #ffeb9c; color: #9c6500",   # yellow
        "high": "background-color: #ffc7ce; color: #9c0006",     # red
    }.get(str(val).lower(), "")
    return color

# Path pattern for all baseline fold prediction files
baseline_pattern = "/content/MLOps-group-10/data/results/baseline_fold*_test.csv"
baseline_files = glob.glob(baseline_pattern)

if baseline_files:
    baseline_df = pd.concat([pd.read_csv(f) for f in baseline_files], ignore_index=True)
    display_cols = [col for col in ["ticker", "company_name", "predicted_score", "predicted_label", "risk_level"] if col in baseline_df.columns]

    print(f"Baseline predictions: {len(baseline_df)} companies (cross-validation)")
    display(
        baseline_df[display_cols].style
        .applymap(style_risk_level, subset=["risk_level"] if "risk_level" in baseline_df.columns else [])
        .format({"predicted_score": "{:.1f}"})
        .set_caption("Baseline Inference Results (Cross-Validation)")
    )

    # Read ground truth
    labels_path = "/content/MLOps-group-10/data/labels/company_labels.csv"
    labels_df = pd.read_csv(labels_path)

    # Merge predictions with ground truth
    merged = baseline_df.merge(labels_df, on="ticker", suffixes=("_pred", "_true"))

    # Figure out which company_name column to use
    company_name_col = "company_name"
    if company_name_col not in merged.columns:
        company_name_col = [col for col in merged.columns if "company_name" in col]
        company_name_col = company_name_col[0] if company_name_col else None

    display_cols = ["ticker"]
    if company_name_col:
        display_cols.append(company_name_col)
    display_cols += ["predicted_label", "label"]

    print(f"Baseline predictions vs Ground Truth: {len(merged)} companies")
    display(
        merged[display_cols]
        .rename(columns={company_name_col: "Company Name", "predicted_label": "Predicted", "label": "Ground Truth"})
        .style.set_caption("Baseline Predictions vs Ground Truth")
    )

    # Example: Calculate accuracy
    accuracy = (merged["predicted_label"] == merged["label"]).mean()
    print(f"Accuracy: {accuracy:.2%}")

else:
    print(f"No baseline prediction files found matching {baseline_pattern}.")

Baseline predictions: 32 companies (cross-validation)


/tmp/ipykernel_4660/4168392740.py:26: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  .applymap(style_risk_level, subset=["risk_level"] if "risk_level" in baseline_df.columns else [])


,ticker,company_name,predicted_score,predicted_label,risk_level
0,AAPL,Apple,20.0,0,low
1,T,AT&T,30.0,0,low
2,GME,GameStop,50.0,0,medium
3,UNH,UnitedHealth Group,20.0,0,low
4,VLO,Valero Energy,30.0,0,low
5,CMCSA,Comcast,30.0,0,low
6,JNJ,Johnson & Johnson,20.0,0,low
7,DAL,Delta Air Lines,30.0,0,low
8,F,Ford Motor Company,35.0,0,low
9,AMZN,Amazon,30.0,0,low


Baseline predictions vs Ground Truth: 32 companies


,ticker,Company Name,Predicted,Ground Truth
0,AAPL,Apple,0,0
1,T,AT&T,0,0
2,GME,GameStop,0,1
3,UNH,UnitedHealth Group,0,0
4,VLO,Valero Energy,0,0
5,CMCSA,Comcast,0,0
6,JNJ,Johnson & Johnson,0,0
7,DAL,Delta Air Lines,0,0
8,F,Ford Motor Company,0,0
9,AMZN,Amazon,0,0


Accuracy: 81.25%


In [14]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


In [19]:
# Verify adapter weights exist for LoRA r32 fold 1
import os
adapter_dir = "/content/MLOps-group-10/data/models/lora_adapter_r32/fold_1/lora_config_r32/final_adapter"
if os.path.isdir(adapter_dir):
    files = os.listdir(adapter_dir)
    print(f"Contents of {adapter_dir}:")
    for f in sorted(files):
        size_mb = os.path.getsize(os.path.join(adapter_dir, f)) / 1e6
        print(f"  {f}  ({size_mb:.2f} MB)")
    if any(f.endswith(".bin") or f.endswith(".safetensors") for f in files):
        print("\n✅ LoRA adapter weights found!")
    else:
        print("\n❌ No adapter weights (.bin or .safetensors) found!")
else:
    print(f"Directory not found: {adapter_dir}")

Contents of /content/MLOps-group-10/data/models/lora_adapter_r32/fold_1/lora_config_r32/final_adapter:
  README.md  (0.01 MB)
  adapter_config.json  (0.00 MB)
  adapter_model.safetensors  (260.10 MB)
  chat_template.jinja  (0.00 MB)
  tokenizer.json  (3.51 MB)
  tokenizer_config.json  (0.00 MB)

✅ LoRA adapter weights found!


---
## Section 5: RAG Inference

Run credit risk inference using **Retrieval-Augmented Generation (RAG)**.

- The FAISS index (built from chunked 10-K text) is used to retrieve the most relevant passages for each company
- Retrieved context is prepended to the prompt before calling Mistral-7B
- This typically improves grounding compared to baseline (less hallucination)

The same quantized model from Section 4 is reused (Colab caches the weights).

In [8]:
for fold in range(1, 6):
    print(f"Running RAG inference for fold {fold} (test split)...")
    !python3 scripts/run_inference_all.py --approach rag --fold {fold} --split test

Running RAG inference for fold 1 (test split)...
04:18:28 [INFO] run_inference_all: Starting inference: approach=rag, mock=False.
04:18:28 [INFO] numexpr.utils: NumExpr defaulting to 2 threads.
04:18:28 [INFO] run_inference_all: Loaded 7 companies for fold 1 split 'test'.
04:18:31 [INFO] run_inference_all: Loading model 'mistralai/Mistral-7B-Instruct-v0.2' (quantize=True).
04:18:41 [INFO] src.models.base_loader: HF_TOKEN found — using authenticated HuggingFace download.
04:18:41 [INFO] src.models.base_loader: Loading tokenizer for 'mistralai/Mistral-7B-Instruct-v0.2'.
04:18:41 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
04:18:41 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/mistralai/Mistral-7B-Instruct-v0.2/63a8b081895390a26e140280378bc85ec8bce07a/config.json "HTTP/1.1 200 OK"
04:18:41 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/mistr

In [11]:
import os
import glob
import pandas as pd
from IPython.display import display

def style_risk_level(val):
    color = {
        "low": "background-color: #c6efce; color: #006100",
        "medium": "background-color: #ffeb9c; color: #9c6500",
        "high": "background-color: #ffc7ce; color: #9c0006",
    }.get(str(val).lower(), "")
    return color

# Path pattern for all RAG fold prediction files
rag_pattern = "/content/MLOps-group-10/data/results/rag_fold*_test.csv"
rag_files = glob.glob(rag_pattern)

if rag_files:
    # Concatenate all folds
    rag_df = pd.concat([pd.read_csv(f) for f in rag_files], ignore_index=True)
    display_cols = [col for col in ["ticker", "company_name", "predicted_score", "predicted_label", "risk_level"] if col in rag_df.columns]

    print(f"RAG predictions: {len(rag_df)} companies (cross-validation)")
    display(
        rag_df[display_cols].style
        .applymap(style_risk_level, subset=["risk_level"] if "risk_level" in rag_df.columns else [])
        .format({"predicted_score": "{:.1f}"})
        .set_caption("RAG Inference Results (Cross-Validation)")
    )

    # Read ground truth
    labels_path = "/content/MLOps-group-10/data/labels/company_labels.csv"
    labels_df = pd.read_csv(labels_path)

    # Merge predictions with ground truth
    merged = rag_df.merge(labels_df, on="ticker", suffixes=("_pred", "_true"))

    # Figure out which company_name column to use
    company_name_col = "company_name"
    if company_name_col not in merged.columns:
        company_name_col = [col for col in merged.columns if "company_name" in col]
        company_name_col = company_name_col[0] if company_name_col else None

    display_cols = ["ticker"]
    if company_name_col:
        display_cols.append(company_name_col)
    display_cols += ["predicted_label", "label"]

    print(f"RAG predictions vs Ground Truth: {len(merged)} companies")
    display(
        merged[display_cols]
        .rename(columns={company_name_col: "Company Name", "predicted_label": "Predicted", "label": "Ground Truth"})
        .style.set_caption("RAG Predictions vs Ground Truth")
    )

    # Example: Calculate accuracy
    accuracy = (merged["predicted_label"] == merged["label"]).mean()
    print(f"Accuracy: {accuracy:.2%}")

else:
    print(f"No RAG prediction files found matching {rag_pattern}.")

RAG predictions: 32 companies (cross-validation)


/tmp/ipykernel_4660/1574262559.py:26: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  .applymap(style_risk_level, subset=["risk_level"] if "risk_level" in rag_df.columns else [])


,ticker,company_name,predicted_score,predicted_label,risk_level
0,JNJ,Johnson & Johnson,50.0,0,medium
1,DAL,Delta Air Lines,50.0,0,medium
2,F,Ford Motor Company,75.0,1,high
3,AMZN,Amazon,75.0,1,high
4,KMX,Carmax,55.0,0,medium
5,CHTR,Charter Communications,75.0,1,high
6,KHC,Kraft Heinz,75.0,1,high
7,AMC,AMC Entertainment,75.0,1,high
8,AVGO,Broadcom,75.0,1,high
9,CI,Cigna Group,75.0,1,high


RAG predictions vs Ground Truth: 32 companies


,ticker,Company Name,Predicted,Ground Truth
0,JNJ,Johnson & Johnson,0,0
1,DAL,Delta Air Lines,0,0
2,F,Ford Motor Company,1,0
3,AMZN,Amazon,1,0
4,KMX,Carmax,0,0
5,CHTR,Charter Communications,1,0
6,KHC,Kraft Heinz,1,1
7,AMC,AMC Entertainment,1,1
8,AVGO,Broadcom,1,0
9,CI,Cigna Group,1,0


Accuracy: 50.00%


---
## Section 6: Generate Fine-Tune Data

The fine-tuning data (`train.jsonl`, `val.jsonl`, `test.jsonl`) is already committed to the repo and should be present.

This cell re-generates it only if the files are missing or you want a fresh split.
The script creates instruction-tuning examples in the format `{instruction, input, output}` from the 10-K sections and labels.

In [6]:
import os

# Regenerate all folds (run this once after updating your data)
os.system("python3 scripts/generate_finetune_data.py")

512

In [7]:
for fold in range(1, 4):  # Use range(1, 6) for 5 folds, or (1, 4) for 3 folds
    print(f"\nFold {fold}:")
    for split in ["train", "val", "test"]:
        path = f"/content/MLOps-group-10/data/finetune/fold_{fold}/{split}.jsonl"
        if os.path.isfile(path):
            with open(path) as f:
                n = sum(1 for line in f if line.strip())
            print(f"  {split:5s}: {n} examples  ({path})")
        else:
            print(f"  {split:5s}: FILE MISSING")


Fold 1:
  train: 20 examples  (/content/MLOps-group-10/data/finetune/fold_1/train.jsonl)
  val  : 5 examples  (/content/MLOps-group-10/data/finetune/fold_1/val.jsonl)
  test : 7 examples  (/content/MLOps-group-10/data/finetune/fold_1/test.jsonl)

Fold 2:
  train: 20 examples  (/content/MLOps-group-10/data/finetune/fold_2/train.jsonl)
  val  : 5 examples  (/content/MLOps-group-10/data/finetune/fold_2/val.jsonl)
  test : 7 examples  (/content/MLOps-group-10/data/finetune/fold_2/test.jsonl)

Fold 3:
  train: 20 examples  (/content/MLOps-group-10/data/finetune/fold_3/train.jsonl)
  val  : 6 examples  (/content/MLOps-group-10/data/finetune/fold_3/val.jsonl)
  test : 6 examples  (/content/MLOps-group-10/data/finetune/fold_3/test.jsonl)


In [8]:
import json

with open("/content/MLOps-group-10/data/finetune/fold_1/train.jsonl") as f:
    first_example = json.loads(f.readline())
print("Sample training example:")
print(json.dumps(first_example, indent=2)[:1000], "..." if len(json.dumps(first_example)) > 1000 else "")

Sample training example:
{
  "text": "<s>[INST] You are a credit risk analyst specializing in SEC 10-K filings. Your task is to assess the credit risk of a company based on excerpts from its annual report.\n\nAnalyze the provided Risk Factors (Item 1A) and Management Discussion & Analysis (Item 7) sections and return a structured JSON assessment.\n\nYour response MUST be valid JSON with exactly these fields:\n{\n  \"predicted_score\": <float 0-100, where 100 = maximum credit risk>,\n  \"risk_level\": <\"low\" | \"medium\" | \"high\">,\n  \"key_signals\": [<list of up to 5 specific risk signals found in the text>],\n  \"citations\": [<list of up to 3 direct quotes from the source text supporting your assessment>],\n  \"rationale\": \"<one paragraph explaining the overall credit risk assessment>\"\n}\n\nScoring guide:\n  0\u201333  \u2192 low risk    (strong financials, stable business, low debt burden)\n  34\u201366 \u2192 medium risk (mixed signals, moderate leverage, some concerns)\n 

---
## Section 7: LoRA Fine-Tuning

Fine-tune Mistral-7B using **LoRA (Low-Rank Adaptation)** with 4-bit quantization via QLoRA.

Three configurations are trained and tested first to compare rank vs. performance vs. compute cost:

| Config | Rank `r` | Alpha | Target modules | W&B run |
|--------|----------|-------|---------------|---------|
| `lora_config_r8.yaml` | 8 | 16 | q, v only | `lora-r8` |
| `lora_config.yaml` | 16 | 32 | q, k, v, o | `lora-sft` |
| `lora_config_r32.yaml` | 32 | 64 | q, k, v, o, up, down | `lora-r32` |

All runs use: 3 epochs, batch size 2, grad accum 4, lr 2e-4, cosine scheduler, bf16.

We found that lora-r32 performs the best so selected it to train. 

In [17]:
!python scripts/train_lora.py --fold 1 --config configs/lora_config_r32.yaml


2026-04-26 11:28:06 — INFO — __main__ — Loading config from /content/MLOps-group-10/configs/lora_config_r32.yaml
2026-04-26 11:28:06 — INFO — __main__ — Model output directory: /content/MLOps-group-10/data/models/lora_adapter_r32/fold_1/lora_config_r32
2026-04-26 11:28:06 — INFO — __main__ — Preparing model for training...
2026-04-26 11:28:15 — INFO — src.models.lora_trainer — Loading tokenizer from 'mistralai/Mistral-7B-Instruct-v0.2'...
2026-04-26 11:28:15 — INFO — httpx — HTTP Request: HEAD https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-26 11:28:15 — INFO — httpx — HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/mistralai/Mistral-7B-Instruct-v0.2/63a8b081895390a26e140280378bc85ec8bce07a/config.json "HTTP/1.1 200 OK"
2026-04-26 11:28:15 — INFO — httpx — HTTP Request: HEAD https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.2/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary R

In [24]:
!python scripts/train_lora.py --fold 2 --config configs/lora_config_r32.yaml


2026-04-26 11:43:22 — INFO — __main__ — Loading config from /content/MLOps-group-10/configs/lora_config_r32.yaml
2026-04-26 11:43:22 — INFO — __main__ — Model output directory: /content/MLOps-group-10/data/models/lora_adapter_r32/fold_2/lora_config_r32
2026-04-26 11:43:22 — INFO — __main__ — Preparing model for training...
2026-04-26 11:43:30 — INFO — src.models.lora_trainer — Loading tokenizer from 'mistralai/Mistral-7B-Instruct-v0.2'...
2026-04-26 11:43:30 — INFO — httpx — HTTP Request: HEAD https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-26 11:43:30 — INFO — httpx — HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/mistralai/Mistral-7B-Instruct-v0.2/63a8b081895390a26e140280378bc85ec8bce07a/config.json "HTTP/1.1 200 OK"
2026-04-26 11:43:30 — INFO — httpx — HTTP Request: HEAD https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.2/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary R

In [30]:
!python scripts/train_lora.py --fold 3 --config configs/lora_config_r32.yaml


2026-04-26 12:02:36 — INFO — __main__ — Loading config from /content/MLOps-group-10/configs/lora_config_r32.yaml
2026-04-26 12:02:36 — INFO — __main__ — Model output directory: /content/MLOps-group-10/data/models/lora_adapter_r32/fold_3/lora_config_r32
2026-04-26 12:02:36 — INFO — __main__ — Preparing model for training...
2026-04-26 12:02:42 — INFO — src.models.lora_trainer — Loading tokenizer from 'mistralai/Mistral-7B-Instruct-v0.2'...
2026-04-26 12:02:42 — INFO — httpx — HTTP Request: HEAD https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-26 12:02:42 — INFO — httpx — HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/mistralai/Mistral-7B-Instruct-v0.2/63a8b081895390a26e140280378bc85ec8bce07a/config.json "HTTP/1.1 200 OK"
2026-04-26 12:02:42 — INFO — httpx — HTTP Request: HEAD https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.2/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary R

In [31]:
!python scripts/train_lora.py --fold 4 --config configs/lora_config_r32.yaml

2026-04-26 12:09:22 — INFO — __main__ — Loading config from /content/MLOps-group-10/configs/lora_config_r32.yaml
2026-04-26 12:09:22 — INFO — __main__ — Model output directory: /content/MLOps-group-10/data/models/lora_adapter_r32/fold_4/lora_config_r32
2026-04-26 12:09:22 — INFO — __main__ — Preparing model for training...
2026-04-26 12:09:32 — INFO — src.models.lora_trainer — Loading tokenizer from 'mistralai/Mistral-7B-Instruct-v0.2'...
2026-04-26 12:09:32 — INFO — httpx — HTTP Request: HEAD https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-26 12:09:32 — INFO — httpx — HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/mistralai/Mistral-7B-Instruct-v0.2/63a8b081895390a26e140280378bc85ec8bce07a/config.json "HTTP/1.1 200 OK"
2026-04-26 12:09:32 — INFO — httpx — HTTP Request: HEAD https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.2/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary R

In [32]:
!python scripts/train_lora.py --fold 5 --config configs/lora_config_r32.yaml

2026-04-26 12:16:06 — INFO — __main__ — Loading config from /content/MLOps-group-10/configs/lora_config_r32.yaml
2026-04-26 12:16:06 — INFO — __main__ — Model output directory: /content/MLOps-group-10/data/models/lora_adapter_r32/fold_5/lora_config_r32
2026-04-26 12:16:06 — INFO — __main__ — Preparing model for training...
2026-04-26 12:16:16 — INFO — src.models.lora_trainer — Loading tokenizer from 'mistralai/Mistral-7B-Instruct-v0.2'...
2026-04-26 12:16:16 — INFO — httpx — HTTP Request: HEAD https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-26 12:16:16 — INFO — httpx — HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/mistralai/Mistral-7B-Instruct-v0.2/63a8b081895390a26e140280378bc85ec8bce07a/config.json "HTTP/1.1 200 OK"
2026-04-26 12:16:16 — INFO — httpx — HTTP Request: HEAD https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.2/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary R

In [ ]:
# OptionalPull training loss curve from W&B and plot locally
try:
    import wandb
    import matplotlib.pyplot as plt

    api = wandb.Api()
    # Find the most recent lora-sft run
    runs = api.runs("credit-risk-detection", filters={"display_name": "lora-sft"})
    runs = sorted(runs, key=lambda r: r.created_at, reverse=True)

    if runs:
        run = runs[0]
        history = run.history(keys=["train/loss", "eval/loss"], pandas=True)

        if not history.empty:
            fig, ax = plt.subplots(figsize=(8, 4))
            if "train/loss" in history.columns:
                ax.plot(history["train/loss"].dropna(), label="Train loss")
            if "eval/loss" in history.columns:
                ax.plot(history["eval/loss"].dropna(), label="Val loss")
            ax.set_xlabel("Step")
            ax.set_ylabel("Loss")
            ax.set_title("LoRA Fine-Tuning Loss Curve")
            ax.legend()
            plt.tight_layout()
            plt.savefig("/content/MLOps-group-10/data/results/lora_loss_curve.png", dpi=150)
            plt.show()
            print(f"Loss curve saved. Run: {run.name} | URL: {run.url}")
        else:
            print("No loss history found in W&B run (may still be syncing).")
    else:
        print("No W&B runs found for lora-sft. Check your W&B dashboard manually.")
except Exception as e:
    print(f"Could not fetch W&B loss curve: {e}")
    print("Check your W&B dashboard at https://wandb.ai")

---
## Section 8: LoRA Inference

Run inference using the **LoRA fine-tuned adapter** on top of Mistral-7B.

The adapter path defaults to `data/models/lora_adapter/final_adapter`. If training produced a different structure, update `ADAPTER_PATH` below.

In [37]:
import os
import subprocess

LORA_CONFIGS = {
    "lora_r32": "data/models/lora_adapter_r32",
}

folds = [1, 2, 3, 4, 5]

for approach, base_dir in LORA_CONFIGS.items():
    for fold in folds:
        # Build possible adapter paths in order of preference
        config_dir = os.path.join(base_dir, f"fold_{fold}", "lora_config_r32")
        final_adapter_dir = os.path.join(config_dir, "final_adapter")
        # Prefer final_adapter if it exists and contains adapter_config.json
        if os.path.isfile(os.path.join(final_adapter_dir, "adapter_config.json")):
            adapter_path = final_adapter_dir
        elif os.path.isfile(os.path.join(config_dir, "adapter_config.json")):
            adapter_path = config_dir
        else:
            print(f"[SKIP] {approach} fold {fold}: adapter_config.json not found in expected locations")
            continue

        print(f"\nRunning inference for {approach} — adapter: {adapter_path} — fold: {fold}")
        result = subprocess.run([
            "python", "scripts/run_inference_all.py",
            "--approach", approach,
            "--adapter-path", adapter_path,
            "--fold", str(fold),
            "--split", "test"
        ], capture_output=True, text=True)
        print("STDOUT:\n", result.stdout)
        print("STDERR:\n", result.stderr)
        if result.returncode != 0:
            print(f"[ERROR] Inference failed for {approach} fold {fold} (exit code {result.returncode})")

print("Inference runs complete.")


Running inference for lora_r32 — adapter: data/models/lora_adapter_r32/fold_1/lora_config_r32/final_adapter — fold: 1
STDOUT:
   PREDICTIONS SUMMARY  (approach: lora_r32)
  Ticker    Company                          Score  Label  Level
  --------------------------------------------------------------------
  JNJ       Johnson & Johnson                 10.0      0  low
  DAL       Delta Air Lines                    0.0      0  low
  F         Ford Motor Company                35.0      0  low
  AMZN      Amazon                             0.0      0  low
  KMX       Carmax                            35.0      0  low
  CHTR      Charter Communications            35.0      0  low
  KHC       Kraft Heinz                       35.0      0  low
  Avg score: 21.4  |  High: 0  Medium: 0  Low: 7
Results saved to: /content/MLOps-group-10/data/results/lora_r32_fold1_test.csv


STDERR:
 12:27:14 [INFO] run_inference_all: Starting inference: approach=lora_r32, mock=False.
12:27:15 [INFO] numexpr.ut

In [44]:
import os
import subprocess

approach = "lora_r32_rag"  # or your actual RAG+LoRA approach name
base_dir = "data/models/lora_adapter_r32"
folds = [1, 2, 3, 4, 5]

for fold in folds:
    config_dir = os.path.join(base_dir, f"fold_{fold}", "lora_config_r32")
    final_adapter_dir = os.path.join(config_dir, "final_adapter")
    # Prefer final_adapter if it exists and contains adapter_config.json
    if os.path.isfile(os.path.join(final_adapter_dir, "adapter_config.json")):
        adapter_path = final_adapter_dir
    elif os.path.isfile(os.path.join(config_dir, "adapter_config.json")):
        adapter_path = config_dir
    else:
        print(f"[SKIP] {approach} fold {fold}: adapter_config.json not found in expected locations")
        continue

    print(f"\nRunning inference for {approach} — adapter: {adapter_path} — fold: {fold}")
    result = subprocess.run([
        "python", "scripts/run_inference_all.py",
        "--approach", approach,
        "--adapter-path", adapter_path,
        "--fold", str(fold),
        "--split", "test"
    ], capture_output=True, text=True)
    print("STDOUT:\n", result.stdout)
    print("STDERR:\n", result.stderr)
    if result.returncode != 0:
        print(f"[ERROR] Inference failed for {approach} fold {fold} (exit code {result.returncode})")

print("Inference runs complete.")

Streaming output truncated to the last 5000 lines.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 541.43it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
13:02:53 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/BAAI/bge-base-en-v1.5/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
13:02:53 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/BAAI/bge-base-en-v1.5/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
13:02:53 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/BAAI/bge-base-en-v1.5/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
13:02:53 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/BAAI/bge-base-en-v1.5/

---
## Section 9: Full Evaluation

Run the complete evaluation pipeline comparing all approaches:

| Approach | Description |
|----------|-------------|
| Baseline | Direct Mistral-7B prompting |
| RAG | Mistral-7B + FAISS retrieval |
| LoRA | Fine-tuned Mistral-7B adapter |
| Altman Z-Score | Classical financial model (benchmark) |

Metrics computed: Accuracy, Precision, Recall, F1, AUC-ROC.

Outputs:
- `data/results/metrics_summary.csv`
- `data/results/roc_curves.png`

In [14]:
# Run full evaluation pipeline — compares all four approaches
# This also generates Altman Z-Score predictions automatically
!python3 scripts/evaluate_folds.py

INFO:evaluate_folds:Loaded and merged 32 rows for baseline
INFO:evaluate_folds:Loaded and merged 32 rows for rag
INFO:evaluate_folds:Loaded and merged 30 rows for lora_r32
INFO:evaluate_folds:Loaded and merged 32 rows for lora_r32_rag
INFO:evaluate_folds:Loaded and merged 9 rows for altman_zscore

  EVALUATION OF ALL FOLDS AND COMPANIES
               auc_roc      f1  precision  recall  accuracy
approach                                                   
baseline        0.8494  0.0000     0.0000  0.0000    0.8125
rag             0.6923  0.4286     0.2727  1.0000    0.5000
lora_r32        0.7014  0.0000     0.0000  0.0000    0.7333
lora_r32_rag    0.5000  0.0000     0.0000  0.0000    0.8125
altman_zscore   0.7778  0.4444     0.3333  0.6667    0.4444

Metrics for baseline:
  auc_roc: 0.8494
  f1: 0.0
  precision: 0.0
  recall: 0.0
  accuracy: 0.8125

Metrics for rag:
  auc_roc: 0.6923
  f1: 0.4286
  precision: 0.2727
  recall: 1.0
  accuracy: 0.5

Metrics for lora_r32:
  auc_roc: 0.7014


In [52]:
!git pull

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 350 bytes | 350.00 KiB/s, done.
From https://github.com/pankti0/MLOps-group-10
   8c531c5..d276168  main       -> origin/main
Updating 8c531c5..d276168
Fast-forward
 scripts/evaluate_folds.py | 1 +
 1 file changed, 1 insertion(+)


In [ ]:
# Load and display the metrics summary table
import pandas as pd
from IPython.display import display
import os

metrics_path = "/content/MLOps-group-10/data/results/metrics_summary.csv"

if os.path.isfile(metrics_path):
    metrics_df = pd.read_csv(metrics_path)
    print("Evaluation Metrics Summary")
    print("=" * 60)

    # Style numeric columns
    numeric_cols = metrics_df.select_dtypes(include="number").columns.tolist()

    def highlight_best(s):
        """Highlight the best value in each numeric column green."""
        if s.name not in numeric_cols:
            return [""] * len(s)
        is_best = s == s.max()
        return ["background-color: #d4edda; font-weight: bold" if v else "" for v in is_best]

    styled = (
        metrics_df.style
        .apply(highlight_best)
        .format({col: "{:.4f}" for col in numeric_cols})
        .set_caption("Comparison of Credit Risk Detection Approaches")
        .set_table_styles([{
            "selector": "th",
            "props": [("background-color", "#343a40"), ("color", "white"), ("font-weight", "bold")]
        }])
    )
    display(styled)
else:
    print(f"Metrics summary not found at {metrics_path}. Check evaluate_folds.py output above.")

In [ ]:
# Display the ROC curves image
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os

roc_path = "/content/MLOps-group-10/data/results/roc_curves.png"

if os.path.isfile(roc_path):
    img = mpimg.imread(roc_path)
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.imshow(img)
    ax.axis("off")
    ax.set_title("ROC Curves — All Approaches", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()
else:
    print(f"ROC curve image not found at {roc_path}.")

In [ ]:
# Show a per-company prediction comparison across all approaches
import pandas as pd
from IPython.display import display
import os

RESULTS_DIR = "/content/MLOps-group-10/data/results"
LABELS_PATH = "/content/MLOps-group-10/data/labels/company_labels.csv"

labels = pd.read_csv(LABELS_PATH)[["ticker", "label", "risk_category"]].rename(
    columns={"label": "true_label", "risk_category": "true_risk"}
)

comparison = labels.copy()

for approach in ["baseline", "rag", "lora_r8", "lora_r16", "lora_r32"]:
    pred_path = os.path.join(RESULTS_DIR, f"{approach}_predictions.csv")
    if os.path.isfile(pred_path):
        df = pd.read_csv(pred_path)[["ticker", "predicted_label", "predicted_score"]]
        df = df.rename(columns={
            "predicted_label": f"{approach}_pred",
            "predicted_score" : f"{approach}_score",
        })
        comparison = comparison.merge(df, on="ticker", how="left")

# Mark correct predictions
for approach in ["baseline", "rag", "lora_r8", "lora_r16", "lora_r32"]:
    col = f"{approach}_pred"
    if col in comparison.columns:
        comparison[f"{approach}_correct"] = (comparison[col] == comparison["true_label"])

print("Per-Company Prediction Comparison")
print("Green cell = correct prediction")

def highlight_correct(val):
    if val is True:
        return "background-color: #d4edda"
    elif val is False:
        return "background-color: #f8d7da"
    return ""

correct_cols = [c for c in comparison.columns if c.endswith("_correct")]
display(
    comparison.style
    .map(highlight_correct, subset=correct_cols)
    .set_caption("Prediction Comparison (green = correct, red = wrong)")
)


---
## Section 10: Download Results

Zip and download the results folder and the LoRA adapter from Colab to your local machine.

- `results.zip` — all prediction CSVs, metrics summary, ROC curves, loss curve
- `lora_adapter.zip` — the trained LoRA adapter weights (for re-use or deployment)

In [42]:
import os
import shutil
from google.colab import files

RESULTS_DIR = "/content/MLOps-group-10/data/results"
ADAPTER_DIR = "/content/MLOps-group-10/data/models/lora_adapter"

# Zip the results directory
results_zip = "/content/results"
print("Zipping data/results/ ...")
shutil.make_archive(results_zip, "zip", RESULTS_DIR)
results_zip_path = results_zip + ".zip"
size_mb = os.path.getsize(results_zip_path) / 1e6
print(f"Created {results_zip_path} ({size_mb:.1f} MB)")

# List contents
print("\nContents:")
for f in sorted(os.listdir(RESULTS_DIR)):
    fpath = os.path.join(RESULTS_DIR, f)
    sz = os.path.getsize(fpath) / 1e3
    print(f"  {f}  ({sz:.1f} KB)")

# Download
print("\nDownloading results.zip ...")
files.download(results_zip_path)

Zipping data/results/ ...
Created /content/results.zip (0.1 MB)

Contents:
  altman_zscore_predictions.csv  (1.8 KB)
  baseline_fold1_test.csv  (11.9 KB)
  baseline_fold2_test.csv  (9.5 KB)
  baseline_fold3_test.csv  (9.6 KB)
  baseline_fold4_test.csv  (9.4 KB)
  baseline_fold5_test.csv  (9.3 KB)
  baseline_predictions.csv  (9.3 KB)
  lora_predictions.csv  (14.6 KB)
  lora_r16_predictions.csv  (13.8 KB)
  lora_r32_fold1_test.csv  (9.0 KB)
  lora_r32_fold2_test.csv  (6.5 KB)
  lora_r32_fold3_test.csv  (7.6 KB)
  lora_r32_fold4_test.csv  (6.3 KB)
  lora_r32_fold5_test.csv  (7.4 KB)
  lora_r32_predictions.csv  (12.3 KB)
  lora_r8_predictions.csv  (17.2 KB)
  metrics_summary.csv  (0.3 KB)
  rag_fold1_test.csv  (7.7 KB)
  rag_fold2_test.csv  (6.4 KB)
  rag_fold3_test.csv  (5.2 KB)
  rag_fold4_test.csv  (5.1 KB)
  rag_fold5_test.csv  (5.7 KB)
  rag_predictions.csv  (5.7 KB)
  roc_curves.png  (110.4 KB)



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [55]:
# Download all LoRA adapters as a single zip
import os
import shutil
from google.colab import files

ADAPTER_DIRS = {
    "lora_r16": "/content/MLOps-group-10/data/models/lora_adapter",
    "lora_r8" : "/content/MLOps-group-10/data/models/lora_adapter_r8",
    "lora_r32": "/content/MLOps-group-10/data/models/lora_adapter_r32",
}

# Bundle all available adapters into one archive
staging = "/content/lora_adapters_all"
os.makedirs(staging, exist_ok=True)

found = []
for name, src in ADAPTER_DIRS.items():
    if os.path.isdir(src):
        shutil.copytree(src, os.path.join(staging, name))
        found.append(name)
        print(f"  Included: {name}")
    else:
        print(f"  [SKIP] {name} — not found at {src}")

if found:
    zip_path = "/content/lora_adapters_all"
    print(f"\nZipping {len(found)} adapter(s)...")
    shutil.make_archive(zip_path, "zip", "/content", "lora_adapters_all")
    size_mb = os.path.getsize(zip_path + ".zip") / 1e6
    print(f"Created lora_adapters_all.zip ({size_mb:.1f} MB)")
    print("Downloading...")
    files.download(zip_path + ".zip")
else:
    print("No adapter directories found. Run Section 7 (LoRA Training) first.")


  [SKIP] lora_r16 — not found at /content/MLOps-group-10/data/models/lora_adapter
  [SKIP] lora_r8 — not found at /content/MLOps-group-10/data/models/lora_adapter_r8
  Included: lora_r32

Zipping 1 adapter(s)...
Created lora_adapters_all.zip (2496.1 MB)
Downloading...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Final summary — print W&B links and key output paths
import os

print("=" * 60)
print("  PIPELINE COMPLETE")
print("=" * 60)

output_files = {
    "Baseline predictions" : "/content/MLOps-group-10/data/results/baseline_predictions.csv",
    "RAG predictions"      : "/content/MLOps-group-10/data/results/rag_predictions.csv",
    "LoRA r=8  predictions": "/content/MLOps-group-10/data/results/lora_r8_predictions.csv",
    "LoRA r=16 predictions": "/content/MLOps-group-10/data/results/lora_r16_predictions.csv",
    "LoRA r=32 predictions": "/content/MLOps-group-10/data/results/lora_r32_predictions.csv",
    "Altman Z-Score"       : "/content/MLOps-group-10/data/results/altman_zscore_predictions.csv",
    "Metrics summary"      : "/content/MLOps-group-10/data/results/metrics_summary.csv",
    "ROC curves"           : "/content/MLOps-group-10/data/results/roc_curves.png",
    "LoRA adapter r=16"    : "/content/MLOps-group-10/data/models/lora_adapter",
    "LoRA adapter r=8"     : "/content/MLOps-group-10/data/models/lora_adapter_r8",
    "LoRA adapter r=32"    : "/content/MLOps-group-10/data/models/lora_adapter_r32",
}

print("\nOutput files:")
for name, path in output_files.items():
    exists = os.path.isfile(path) or os.path.isdir(path)
    status = "OK" if exists else "MISSING"
    print(f"  [{status:7s}] {name}")

print("\nW&B Dashboard:")
print("  https://wandb.ai → project: credit-risk-detection")
print("  Runs: lora-r8, lora-sft, lora-r32 (training) | evaluation")
print("=" * 60)
